# 04_GSE193371_qc_selection

**Thesis Methods section(s): 4.1.1, 4.1.2, 4.1.3**

**Reads:** GSE193371 raw files (GEO).

**Writes:** GSE193371_CD8_STRICT_postQC_scvi_rawcounts.h5ad

**Notes:** Cells were already CD8-restricted by the original authors; no marker rule applied. Contains an exploratory per-dataset scVI model NOT reported in the thesis.

Input data are not included in this repository. Set `DATA_ROOT` below to a local folder
holding the GEO downloads; see `README.md` for accessions and the expected layout.


In [ ]:
# Root folder for input data (NOT included in this repository).
# Set the DATA_ROOT environment variable, or edit the fallback below.
import os
DATA_ROOT = os.environ.get("DATA_ROOT", "data")


In [ ]:
import os
import scanpy as sc
import scvi
import matplotlib.pyplot as plt

# 📁 Path to the parent directory containing all patient folders
parent_dir = f"{DATA_ROOT}/Single-cell RNA seq/Collaboration with FB/GSE193371 - HGSOC/Raw files"  # 🔁 CHANGE this to your actual path

In [ ]:
# 🧠 Define function to load each folder as AnnData
def load_10x_folder(folder_name):
    path = os.path.join(parent_dir, folder_name) # Create the full path to the folder
    adata = sc.read_10x_mtx(path, var_names="gene_symbols", cache=True) # Load the single-cell data
    adata.obs["sample_id"] = folder_name # Add a column to store which patient the data is from
    if "T" in folder_name:
        patient_id = folder_name.split("-")[0]
        adata.obs["patient_id"] = patient_id # Extract patient ID from the folder name
    if "TRM" in folder_name: # If the folder name contains "TIL"
        adata.obs["source"] = "TRM" # Mark the source as Tumor-Infiltrating Lymphocytes
    elif "ReCir" in folder_name: # If the folder name contains "PBL"
        adata.obs["source"] = "ReCir" # Mark the source as Peripheral Blood Lymphocytes
    return adata # Return the loaded data

In [ ]:
# 📂 All folder names
folders = [
    "T021518-TRM", "T022719-TRM", "T082219-TRM", 
    "T021518-ReCir", "T022719-ReCir", "T082219-ReCir"
] # 3 TRM and 3 ReCir folders

# 🧬 Load and store all AnnData objects in a list
all_adata = [load_10x_folder(f) for f in folders] # This gives you 6 AnnData objects

# 🔗 Merge all into one AnnData object
GSE193371 = all_adata[0].concatenate(*all_adata[1:], batch_key="sample_id", batch_categories=folders)

# ✅ Now adata.obs contains:
# - `adata.obs["patient"]` tells you which patient each cell came from
# - `adata.obs["source"]` tells you whether it came from blood (PBL) or tumor (TIL)

In [ ]:
GSE193371.obs.columns

In [ ]:
GSE193371.obs["sample_id"].value_counts()

In [ ]:
GSE193371.obs["patient_id"].value_counts()

In [ ]:
GSE193371.obs["source"].value_counts()

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# 📐 Count cells per patient and source
counts = GSE193371.obs.groupby(["patient_id", "source"]).size().reset_index(name="cell_count")

# 🎨 Set the style
sns.set(style="whitegrid")

# 📊 Plot
plt.figure(figsize=(10, 6))
sns.barplot(data=counts, x="patient_id", y="cell_count", hue="source", palette="viridis")

# ✨ Decorations
plt.title("Cell Counts per Patient and Sample Type (TRM vs ReCir)", fontsize=14)
plt.xlabel("Patient ID", fontsize=12)
plt.ylabel("Number of Cells", fontsize=12)
plt.legend(title="Sample Source")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Quality Control (QC)
import scanpy as sc
import matplotlib.pyplot as plt

# Calculate QC metrics
GSE193371.var['mt'] = GSE193371.var_names.str.upper().str.startswith('MT-')  # Ensures uppercase compatibility
sc.pp.calculate_qc_metrics(GSE193371, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)

# Plot QC metrics before filtering
fig, axs = plt.subplots(1, 3, figsize=(15, 4))

# Total counts per cell
axs[0].hist(GSE193371.obs['total_counts'], bins=50, color='skyblue')
axs[0].set_title('Total Counts per Cell')
axs[0].set_xlabel('Total Counts')
axs[0].set_ylabel('Number of Cells')

# Number of genes per cell
axs[1].hist(GSE193371.obs['n_genes_by_counts'], bins=50, color='lightgreen')
axs[1].set_title('Number of Genes per Cell')
axs[1].set_xlabel('Genes')
axs[1].set_ylabel('Number of Cells')

# Percent mitochondrial genes
axs[2].hist(GSE193371.obs['pct_counts_mt'], bins=50, color='salmon')
axs[2].set_title('% Mitochondrial Genes')
axs[2].set_xlabel('% MT Genes')
axs[2].set_ylabel('Number of Cells')

plt.tight_layout()
plt.show()

In [ ]:
# You can now visually decide thresholds.
# Here's one possible filtering step based on common practice:
sc.pp.filter_cells(GSE193371, min_genes=200)
sc.pp.filter_genes(GSE193371, min_cells=3)
GSE193371 = GSE193371[GSE193371.obs.pct_counts_mt < 10, :]  # Modify threshold based on histogram

In [ ]:
# 📌 Save raw version for scVI
GSE193371_raw_scVI = GSE193371.copy()

In [ ]:
# 📍 UMAP before scVI (PCA-based)
# ----------------------------------

sc.pp.normalize_total(GSE193371, target_sum=1e4)
sc.pp.log1p(GSE193371)
sc.pp.highly_variable_genes(GSE193371, n_top_genes=2000, subset=True)
sc.pp.scale(GSE193371, max_value=10)
sc.tl.pca(GSE193371, svd_solver='arpack')

In [ ]:
# 🔍 Elbow plot to choose number of PCs
import matplotlib.pyplot as plt
import numpy as np

explained = GSE193371.uns["pca"]["variance_ratio"]
plt.figure(figsize=(6, 4))
plt.plot(range(1, len(explained) + 1), explained, marker='o')
plt.xlabel('Principal Component')
plt.ylabel('Explained Variance Ratio')
plt.title('PCA Elbow Plot')
plt.grid(True)
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

explained = GSE193371.uns["pca"]["variance_ratio"]
cumulative = np.cumsum(explained)

plt.figure(figsize=(7, 5))
plt.plot(range(1, len(explained) + 1), explained, marker='o', label='Individual variance')
plt.plot(range(1, len(cumulative) + 1), cumulative, marker='x', linestyle='--', label='Cumulative variance')

plt.axhline(y=0.05, color='red', linestyle='--', label='5% variance threshold')
plt.axvline(x=np.argmax(cumulative >= 0.9) + 1, color='green', linestyle='--', label='90% cumulative variance')

plt.xlabel('Principal Component')
plt.ylabel('Explained Variance Ratio')
plt.title('PCA Elbow Plot')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
GSE193371_Merged = sc.read(f"{DATA_ROOT}/scVI/GSE193371_umap_before_scvi.h5ad")

In [ ]:
# Dimensionality reduction & clustering
sc.pp.neighbors(GSE193371, n_neighbors=15, n_pcs=10)
sc.tl.umap(GSE193371)
sc.tl.leiden(GSE193371, resolution=0.5)

In [ ]:
# UMAP plot (PCA-based)
sc.pl.umap(
    GSE193371,
    color=["leiden"],
    title=["Before scVI: Clusters"]
)

In [ ]:
# UMAP plot (PCA-based)
sc.pl.umap(
    GSE193371,
    color=["patient_id"],
    title=["Before scVI: Patient"],
)

In [ ]:
# UMAP plot (PCA-based)
sc.pl.umap(
    GSE193371,
    color=["source"],
    title=["Before scVI: Sample Type"],
)

In [ ]:
# UMAP plot (PCA-based)
sc.pl.umap(
    GSE193371,
    color=["sample_id"],
    title=["Before scVI: Sample ID"],
)

In [ ]:
# Optionally save the file
GSE193371.write(
    f"{DATA_ROOT}/scVI/GSE193371_PCA_based.h5ad"
)

In [ ]:
# Load the saved AnnData object
GSE193371 = sc.read(f"{DATA_ROOT}/scVI/GSE138720_PCA-based.h5ad")

In [ ]:
# Optionally save the file
GSE193371_raw_scVI.write(
    f"{DATA_ROOT}/scVI/GSE193371_raw.h5ad"
)

In [ ]:
# Load the saved AnnData object
GSE193371_raw_scVI = sc.read(f"{DATA_ROOT}/scVI/GSE193371_raw.h5ad")

In [ ]:
import os
import scvi

# Define save path
model_path = "scvi_model/GSE193371_scvi_model"

# Make sure the folder exists
os.makedirs("scvi_model", exist_ok=True)

# Set up AnnData for scVI
scvi.model.SCVI.setup_anndata(GSE193371_raw_scVI, batch_key="sample_id")

# Load or train
if os.path.exists(os.path.join(model_path, "model.pt")):
    model = scvi.model.SCVI.load(model_path, GSE193371_raw_scVI)
    print("✅ Loaded saved scVI model.")
else:
    model = scvi.model.SCVI(GSE193371_raw_scVI)
    model.train(max_epochs=100)
    model.save(model_path, overwrite=True)
    print("✅ Trained and saved scVI model.")

In [ ]:
# Plot scVI Training Loss Curve
# ----------------------------------
import matplotlib.pyplot as plt

# Get history directly from trained model
history = model.history
print("Available keys in history:", history.keys())
elbo = history["elbo_train"]

# Plot
plt.plot(range(1, len(elbo) + 1), elbo, marker='o')
plt.xlabel("Epoch")
plt.ylabel("Training ELBO Loss")
plt.title("scVI Training Loss")
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# Get scVI latent space and compute UMAP
GSE193371_raw_scVI.obsm["X_scVI"] = model.get_latent_representation()

In [ ]:
# Optionally save the file
GSE193371_raw_scVI.write(
    f"{DATA_ROOT}/scVI/GSE193371_scVI_based.h5ad"
)

In [ ]:
# Load the saved AnnData object
import scanpy as sc
GSE138720_raw_scVI = sc.read(f"{DATA_ROOT}/scVI/GSE138720_raw_after_scvi.h5ad")

In [ ]:
# UMAP & Clustering on scVI Latent Space
# --------------------------------------------

# Step 1: Compute neighborhood graph using the scVI latent space
sc.pp.neighbors(GSE193371_raw_scVI, use_rep="X_scVI", n_neighbors=15)

# Step 2: Run UMAP (2D)
sc.tl.umap(GSE193371_raw_scVI, n_components=2)

# Step 3: Leiden clustering
sc.tl.leiden(GSE193371_raw_scVI, resolution=0.5)

# Step 4 (Optional): Check number of clusters
n_clusters = GSE193371_raw_scVI.obs["leiden"].nunique()
print(f"✅ Leiden clustering at resolution 0.5 → {n_clusters} clusters")

In [ ]:
# UMAP plot (scVI-based)
sc.pl.umap(
    GSE193371_raw_scVI,
    color=["leiden"],
    title=["After scVI: Clusters"],
)

In [ ]:
# UMAP plot (scVI-based)
sc.pl.umap(
    GSE193371_raw_scVI,
    color=["patient_id"],
    title=["After scVI: Patient ID"],
)

In [ ]:
# UMAP plot (scVI-based)
sc.pl.umap(
    GSE193371_raw_scVI,
    color=["source"],
    title=["After scVI: Source (TRM vs ReCir)"],
)

In [ ]:
# UMAP plot (scVI-based)
sc.pl.umap(
    GSE193371_raw_scVI,
    color=["sample_id"],
    title=["After scVI: Sample ID"],
)